In [52]:
from fantasy_football.storage.database import load_player_match, get_connection

import duckdb
import polars as pl

In [53]:
conn = get_connection()

In [54]:
player_match = load_player_match()
player_match.head()

season,gw,element,opponent,is_home,minutes,total_points
str,i64,i64,i64,bool,i64,i64
"""2016-17""",1,6,9,true,90,0
"""2016-17""",1,7,9,true,0,0
"""2016-17""",1,11,9,true,90,6
"""2016-17""",1,13,9,true,90,5
"""2016-17""",1,14,9,true,0,0


In [55]:
from sklearn.model_selection import train_test_split
# get a random selection of buckets
# come up with metrics 
# score random baseline
# add rolling minutes as feature, and score this 
# reassess on the features you have

In [56]:
def create_buckets(player_data: pl.DataFrame, column_to_bucket: str = "minutes") -> pl.DataFrame:
    player_data = player_data.with_columns(
        pl.when(pl.col(column_to_bucket) == 0)
        .then(pl.lit("0_minutes"))
        .when(pl.col(column_to_bucket) < 60)
        .then(pl.lit("1_to_59_minutes"))
        .otherwise(pl.lit("60_minutes_plus"))
        .alias("minutes_bucket")
    )
    return player_data

In [57]:
player_match = create_buckets(player_match)
player_match.head()

season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket
str,i64,i64,i64,bool,i64,i64,str
"""2016-17""",1,6,9,true,90,0,"""60_minutes_plus"""
"""2016-17""",1,7,9,true,0,0,"""0_minutes"""
"""2016-17""",1,11,9,true,90,6,"""60_minutes_plus"""
"""2016-17""",1,13,9,true,90,5,"""60_minutes_plus"""
"""2016-17""",1,14,9,true,0,0,"""0_minutes"""


In [58]:
from sklearn.metrics import classification_report
from random import choice

buckets = ["0_minutes", "1_to_59_minutes", "60_minutes_plus"]

random_baseline = [choice(buckets) for _ in range(len(player_match))]

cr = classification_report(player_match["minutes_bucket"], random_baseline)
print(cr)


                 precision    recall  f1-score   support

      0_minutes       0.58      0.34      0.42    105036
1_to_59_minutes       0.12      0.34      0.18     21651
60_minutes_plus       0.31      0.34      0.32     56741

       accuracy                           0.34    183428
      macro avg       0.34      0.34      0.31    183428
   weighted avg       0.44      0.34      0.36    183428



In [59]:
train_seasons = ["2022-23", "2023-24", "2024-25"]
test_season = "2025-26"

train = player_match.filter(pl.col("season").is_in(train_seasons))
test = player_match.filter(pl.col("season") == test_season)
test_y = create_buckets(test)
test_y.head()


season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket
str,i64,i64,i64,bool,i64,i64,str
"""2025-26""",1,1,14,false,90,10,"""60_minutes_plus"""
"""2025-26""",1,2,14,false,0,0,"""0_minutes"""
"""2025-26""",1,3,14,false,0,0,"""0_minutes"""
"""2025-26""",1,4,14,false,0,0,"""0_minutes"""
"""2025-26""",1,5,14,false,90,6,"""60_minutes_plus"""


In [60]:
def create_rolling_minutes(player_data: pl.DataFrame, window: int = 3) -> pl.DataFrame:
    # Rolling mean of each player's *prior* gameweeks.
    #   - sort by element, season, gw then .over("element") so every window stays
    #     inside one player in chronological order. (The raw frame is ordered
    #     season, gw, element, so a plain rolling_mean blends three players together.)
    #     polars 0.20.x has no order_by= on .over(), hence the explicit sort.
    #   - .shift(1) drops the current match, so we never leak this week's minutes
    #     into the feature predicting this week's bucket.
    #   - _orig restores the original row order so downstream frames stay aligned.
    return (
        player_data
        .with_row_index("_orig")
        .sort(["element", "season", "gw"])
        .with_columns(
            pl.col("minutes")
            .shift(1)
            .rolling_mean(window_size=window, min_periods=1)
            .over("element")
            .alias("rolling_minutes")
        )
        .sort("_orig")
        .drop("_orig")
    )

train = create_rolling_minutes(train)
train.head()

season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket,rolling_minutes
str,i64,i64,i64,bool,i64,i64,str,f64
"""2022-23""",1,1,7,false,0,0,"""0_minutes""",null
"""2022-23""",1,2,12,true,0,0,"""0_minutes""",null
"""2022-23""",1,3,7,false,90,2,"""60_minutes_plus""",null
"""2022-23""",1,4,7,false,0,0,"""0_minutes""",null
"""2022-23""",1,5,7,false,0,0,"""0_minutes""",null


In [61]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median").set_output(transform="polars")
train = train.with_columns(imputer.fit_transform(train.select("rolling_minutes")))
train.head()



season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket,rolling_minutes
str,i64,i64,i64,bool,i64,i64,str,f64
"""2022-23""",1,1,7,false,0,0,"""0_minutes""",1.0
"""2022-23""",1,2,12,true,0,0,"""0_minutes""",1.0
"""2022-23""",1,3,7,false,90,2,"""60_minutes_plus""",1.0
"""2022-23""",1,4,7,false,0,0,"""0_minutes""",1.0
"""2022-23""",1,5,7,false,0,0,"""0_minutes""",1.0


In [62]:
test = create_rolling_minutes(test)
# Keep rolling_minutes as a column (don't replace the whole frame) so the bucket
# step below still has season/gw/element/minutes alongside it. Imputer was fit on
# train only — correct, no test leakage.
test = test.with_columns(imputer.transform(test.select("rolling_minutes")))
test.head()

season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket,rolling_minutes
str,i64,i64,i64,bool,i64,i64,str,f64
"""2025-26""",1,1,14,false,90,10,"""60_minutes_plus""",1.0
"""2025-26""",1,2,14,false,0,0,"""0_minutes""",1.0
"""2025-26""",1,3,14,false,0,0,"""0_minutes""",1.0
"""2025-26""",1,4,14,false,0,0,"""0_minutes""",1.0
"""2025-26""",1,5,14,false,90,6,"""60_minutes_plus""",1.0


In [63]:
train_predictions = create_buckets(train, "rolling_minutes")
train_predictions.head()

season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket,rolling_minutes
str,i64,i64,i64,bool,i64,i64,str,f64
"""2022-23""",1,1,7,false,0,0,"""1_to_59_minutes""",1.0
"""2022-23""",1,2,12,true,0,0,"""1_to_59_minutes""",1.0
"""2022-23""",1,3,7,false,90,2,"""1_to_59_minutes""",1.0
"""2022-23""",1,4,7,false,0,0,"""1_to_59_minutes""",1.0
"""2022-23""",1,5,7,false,0,0,"""1_to_59_minutes""",1.0


In [64]:
test_predictions = create_buckets(test, "rolling_minutes")
test_predictions.head()


season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket,rolling_minutes
str,i64,i64,i64,bool,i64,i64,str,f64
"""2025-26""",1,1,14,false,90,10,"""1_to_59_minutes""",1.0
"""2025-26""",1,2,14,false,0,0,"""1_to_59_minutes""",1.0
"""2025-26""",1,3,14,false,0,0,"""1_to_59_minutes""",1.0
"""2025-26""",1,4,14,false,0,0,"""1_to_59_minutes""",1.0
"""2025-26""",1,5,14,false,90,6,"""1_to_59_minutes""",1.0


In [65]:
rolling_points_score = classification_report(test_predictions["minutes_bucket"], test_y["minutes_bucket"])
print(rolling_points_score)

                 precision    recall  f1-score   support

      0_minutes       0.80      0.96      0.87     15207
1_to_59_minutes       0.67      0.32      0.43      7726
60_minutes_plus       0.69      0.79      0.74      6814

       accuracy                           0.75     29747
      macro avg       0.72      0.69      0.68     29747
   weighted avg       0.74      0.75      0.73     29747



In [ ]:
# Value-based features for *every* (season, gw, element) in one pass, instead of
# one DuckDB round-trip per player. Each metric is a window over player_week:
#   value_vs_team   - share of the club's gw budget tied up in this player
#   pos_value_rank  - 1 = most expensive in his position at the club;
#                     (rank - 1) is also "how many same-position teammates cost more"
#   players_same_pos- squad depth in his position at the club
def load_value_features(conn: duckdb.DuckDBPyConnection) -> pl.DataFrame:
    query = """
    SELECT
        season,
        gw,
        element,
        team,
        position,
        value,
        value::DOUBLE / SUM(value) OVER (PARTITION BY season, gw, team)          AS value_vs_team,
        RANK()  OVER (PARTITION BY season, gw, team, position ORDER BY value DESC) AS pos_value_rank,
        COUNT(*) OVER (PARTITION BY season, gw, team, position)                   AS players_same_pos
    FROM player_week
    """
    return conn.execute(query).pl()


value_features = load_value_features(conn)
value_features.head()

In [67]:
# Attach the value features once, on the key. Do this BEFORE the train/test split
# (move it just under the create_buckets cell) so both splits carry the columns.
player_match = player_match.join(
    value_features,
    on=["season", "gw", "element"],
    how="left",
)
player_match.head()

/var/folders/ws/c0kbfc596sgcz0f3y4dzqtyc0000gn/T/ipykernel_48828/4247086550.py:3: DeprecationWarning: The default coalesce behavior of left join will change to `False` in the next breaking release. Pass `coalesce=True` to keep the current behavior and silence this warning.
  player_match = player_match.join(


season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket,position,value,value_vs_team,pos_value_rank,players_same_pos
str,i64,i64,i64,bool,i64,i64,str,str,i64,f64,i64,i64
"""2016-17""",1,6,9,true,90,0,"""60_minutes_plus""","""DEF""",65,0.005745,1,76
"""2016-17""",1,7,9,true,0,0,"""0_minutes""","""DEF""",50,0.004419,28,76
"""2016-17""",1,11,9,true,90,6,"""60_minutes_plus""","""DEF""",45,0.003977,49,76
"""2016-17""",1,13,9,true,90,5,"""60_minutes_plus""","""MID""",75,0.006628,11,71
"""2016-17""",1,14,9,true,0,0,"""0_minutes""","""MID""",95,0.008396,2,71


In [ ]:
# Attach FPL's point-in-time chance_of_playing_this_round, on the same key, before
# the split so both train and test carry it. add_chance_of_playing left-joins the
# player_availability table and defaults any uncovered (season, gw, element) to 100
# (= no injury doubt). The value is the deadline snapshot for *this* gameweek, so
# it's known before kickoff — a legitimate predictor, no leakage.
from fantasy_football.storage.database import load_player_availability
from fantasy_football.features.availability import (
    add_chance_of_playing,
    add_positional_availability,
)
from fantasy_football.extraction.availability import load_player_availability_data

# player_availability is sourced from the Randdalf/fplcache snapshots (2022-23 on).
# Populate it once if empty; the first run is slow (~a bootstrap snapshot per
# gameweek per season), reruns are cheap (immutable seasons skip, current upserts).
if load_player_availability(conn).is_empty():
    load_player_availability_data(conn)

availability = load_player_availability(conn)
player_match = add_chance_of_playing(player_match, availability)

# Count fit same-position rivals. Runs after add_chance_of_playing so null
# chances are already filled to 100 (= fit). Needs team/position/value, all
# present from the value-features join above.
player_match = add_positional_availability(player_match)
player_match.head()

In [ ]:
features = [
    "position",
    "value",
    "value_vs_team",
    "pos_value_rank",
    "players_same_pos",
    "chance_of_playing_this_round",
    "fit_rivals_same_pos",
    "fit_rivals_ahead",
]
train_X = player_match.filter(pl.col("season").is_in(train_seasons)).select(features)
train_y = player_match.filter(pl.col("season").is_in(train_seasons)).select("minutes_bucket")

test_X = player_match.filter(pl.col("season") == test_season).select(features)
test_y = player_match.filter(pl.col("season") == test_season).select("minutes_bucket")


In [69]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False).set_output(transform="polars")
train_pos = ohe.fit_transform(train_X.select("position"))
test_pos = ohe.transform(test_X.select("position"))

train_X = train_X.drop("position")
train_X = pl.concat([train_X, train_pos], how="horizontal")

test_X = test_X.drop("position")
test_X = pl.concat([test_X, test_pos], how="horizontal")

train_X.head()



value,value_vs_team,pos_value_rank,players_same_pos,position_AM,position_DEF,position_FWD,position_GK,position_MID
i64,f64,i64,i64,f64,f64,f64,f64,f64
45,0.030612,4,10,0.0,1.0,0.0,0.0,0.0
45,0.033582,1,3,0.0,0.0,0.0,1.0,0.0
50,0.034014,7,14,0.0,0.0,0.0,0.0,1.0
45,0.030612,12,14,0.0,0.0,0.0,0.0,1.0
45,0.030612,4,10,0.0,1.0,0.0,0.0,0.0


In [70]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(train_X, train_y)

pred = lr.predict(test_X)

cr = classification_report(test_y["minutes_bucket"], pred)
print(cr)


/Users/jamie/personal/fantasy_football/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/jamie/personal/fantasy_football/.venv/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/jamie/personal/fantasy_football/.venv/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/jamie/personal/fantasy_football/.venv/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # n

                 precision    recall  f1-score   support

      0_minutes       0.70      0.89      0.78     18255
1_to_59_minutes       0.00      0.00      0.00      3679
60_minutes_plus       0.51      0.41      0.46      7813

       accuracy                           0.66     29747
      macro avg       0.40      0.43      0.41     29747
   weighted avg       0.56      0.66      0.60     29747



/Users/jamie/personal/fantasy_football/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/jamie/personal/fantasy_football/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/jamie/personal/fantasy_football/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/jamie/personal/fantasy_football/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

In [71]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(train_X, train_y)

pred = rf.predict(test_X)

cr = classification_report(test_y["minutes_bucket"], pred)
print(cr)


/Users/jamie/personal/fantasy_football/.venv/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


                 precision    recall  f1-score   support

      0_minutes       0.72      0.79      0.76     18255
1_to_59_minutes       0.20      0.10      0.13      3679
60_minutes_plus       0.47      0.48      0.48      7813

       accuracy                           0.62     29747
      macro avg       0.46      0.46      0.45     29747
   weighted avg       0.59      0.62      0.60     29747



## Error analysis — confidently wrong, far-off predictions

`pred` aligns row-for-row with `test_X`, which is `player_match` filtered to the
test season in original order. So we can glue predictions back onto the test rows
by position, attach `predict_proba` confidence, score each row by how many buckets
it missed by, then join `player_week` for names to see *who* the model got wrong.

In [72]:
# Ordinal rank of each bucket so we can measure "how far off" a miss was.
# Built with when/then (not replace_strict) to stay safe on polars 0.20.x.
def bucket_ord(col: str) -> pl.Expr:
    return (
        pl.when(pl.col(col) == "0_minutes").then(0)
        .when(pl.col(col) == "1_to_59_minutes").then(1)
        .otherwise(2)
    )

# Pick the model to analyse: rf is the most recent fit; swap to lr if you prefer.
model = rf
test_rows = player_match.filter(pl.col("season") == test_season)

# Confidence the model placed on the class it actually predicted.
proba = model.predict_proba(test_X)
pred_confidence = proba.max(axis=1)

errors = (
    test_rows
    # keys + the exact features the model trained on (`features`) + the target,
    # so every row shows the inputs that drove its (mis)prediction.
    .select(
        "season", "gw", "element",
        *features,
        "minutes", "minutes_bucket",
    )
    .with_columns(
        pl.Series("predicted_bucket", pred),
        pl.Series("predicted_confidence", pred_confidence),
    )
    .with_columns(
        (bucket_ord("predicted_bucket") - bucket_ord("minutes_bucket"))
        .abs()
        .alias("bucket_distance")
    )
)
errors.head()

season,gw,element,position,value,value_vs_team,pos_value_rank,players_same_pos,minutes,minutes_bucket,predicted_bucket,predicted_confidence,bucket_distance
str,i64,i64,str,i64,f64,i64,i64,i64,str,str,f64,i32
"""2025-26""",1,1,"""GK""",60,0.02893,1,4,90,"""60_minutes_plus""","""60_minutes_plus""",0.61,0
"""2025-26""",1,2,"""GK""",41,0.019769,2,4,0,"""0_minutes""","""0_minutes""",0.84,0
"""2025-26""",1,3,"""GK""",40,0.019286,3,4,0,"""0_minutes""","""0_minutes""",1.0,0
"""2025-26""",1,4,"""GK""",39,0.018804,4,4,0,"""0_minutes""","""0_minutes""",1.0,0
"""2025-26""",1,5,"""DEF""",62,0.029894,2,14,90,"""60_minutes_plus""","""60_minutes_plus""",0.751333,0


In [73]:
# Join player identity. player_week carries name/team per (season, gw, element),
# so we learn *who* each row is and what club they were at that gameweek.
from fantasy_football.storage.database import load_player_week

names = load_player_week().select("season", "gw", "element", "name", "team")
errors = errors.join(names, on=["season", "gw", "element"], how="left")

# Worst misses first: biggest bucket gap, and among those the most confident calls.
# Show the trained features alongside so you can see which inputs misled the model.
worst = (
    errors
    .filter(pl.col("predicted_bucket") != pl.col("minutes_bucket"))
    .sort(["bucket_distance", "predicted_confidence"], descending=True)
    .select(
        "name", "team", "gw",
        *features,
        "minutes", "minutes_bucket", "predicted_bucket",
        "predicted_confidence", "bucket_distance",
    )
)
worst.head(30)

/var/folders/ws/c0kbfc596sgcz0f3y4dzqtyc0000gn/T/ipykernel_48828/1865781935.py:6: DeprecationWarning: The default coalesce behavior of left join will change to `False` in the next breaking release. Pass `coalesce=True` to keep the current behavior and silence this warning.
  errors = errors.join(names, on=["season", "gw", "element"], how="left")


name,team,gw,position,value,value_vs_team,pos_value_rank,players_same_pos,minutes,minutes_bucket,predicted_bucket,predicted_confidence,bucket_distance
str,str,i64,str,i64,f64,i64,i64,i64,str,str,f64,i32
"""James Hill""","""Bournemouth""",4,"""DEF""",40,0.020975,8,13,86,"""60_minutes_plus""","""0_minutes""",1.0,2
"""Robin Roefs""","""Sunderland""",31,"""GK""",48,0.020557,1,5,0,"""0_minutes""","""60_minutes_plus""",1.0,2
"""Lukasz Fabianski""","""West Ham""",22,"""GK""",45,0.023709,1,5,0,"""0_minutes""","""60_minutes_plus""",1.0,2
"""José Malheiro de Sá""","""Wolves""",1,"""GK""",42,0.023918,2,4,90,"""60_minutes_plus""","""0_minutes""",1.0,2
"""Martin Dúbravka""","""Burnley""",5,"""GK""",40,0.021494,2,4,90,"""60_minutes_plus""","""0_minutes""",1.0,2
…,…,…,…,…,…,…,…,…,…,…,…,…
"""Martin Dúbravka""","""Burnley""",3,"""GK""",40,0.021379,2,4,90,"""60_minutes_plus""","""0_minutes""",1.0,2
"""Martin Dúbravka""","""Burnley""",14,"""GK""",40,0.021751,2,4,90,"""60_minutes_plus""","""0_minutes""",1.0,2
"""John Victor Maciel Furtado""","""Nott'm Forest""",16,"""GK""",40,0.02187,2,5,90,"""60_minutes_plus""","""0_minutes""",1.0,2


In [74]:
def bucket_ord(col):
    return (
        pl.when(pl.col(col) == "0_minutes").then(0)
        .when(pl.col(col) == "1_to_59_minutes").then(1)
        .otherwise(2)
    )

model = rf
test_rows = player_match.filter(pl.col("season") == test_season)
proba = model.predict_proba(test_X)
pred_confidence = proba.max(axis=1)

errors = (
    test_rows
    .select("season", "gw", "element", "minutes", "minutes_bucket", *features)
    .with_columns(
        pl.Series("predicted_bucket", pred),
        pl.Series("predicted_confidence", pred_confidence),
    )
    .with_columns(
        (bucket_ord("predicted_bucket") - bucket_ord("minutes_bucket")).abs().alias("bucket_distance")
    )
)

names = load_player_week().select("season", "gw", "element", "name", "team")
errors = errors.join(names, on=["season", "gw", "element"], how="left")

worst = (
    errors.filter(pl.col("predicted_bucket") != pl.col("minutes_bucket"))
    .sort(["bucket_distance", "predicted_confidence"], descending=True)
    .select("name","team","position","gw","value","minutes","minutes_bucket","predicted_bucket","predicted_confidence","bucket_distance")
)
print(errors.shape, "errors rows; worst preview:")
print(worst.head(10))

(29747, 15) errors rows; worst preview:
shape: (10, 10)
┌────────────┬────────────┬──────────┬─────┬───┬────────────┬────────────┬────────────┬────────────┐
│ name       ┆ team       ┆ position ┆ gw  ┆ … ┆ minutes_bu ┆ predicted_ ┆ predicted_ ┆ bucket_dis │
│ ---        ┆ ---        ┆ ---      ┆ --- ┆   ┆ cket       ┆ bucket     ┆ confidence ┆ tance      │
│ str        ┆ str        ┆ str      ┆ i64 ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│            ┆            ┆          ┆     ┆   ┆ str        ┆ str        ┆ f64        ┆ i32        │
╞════════════╪════════════╪══════════╪═════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ James Hill ┆ Bournemout ┆ DEF      ┆ 4   ┆ … ┆ 60_minutes ┆ 0_minutes  ┆ 1.0        ┆ 2          │
│            ┆ h          ┆          ┆     ┆   ┆ _plus      ┆            ┆            ┆            │
│ Robin      ┆ Sunderland ┆ GK       ┆ 31  ┆ … ┆ 0_minutes  ┆ 60_minutes ┆ 1.0        ┆ 2          │
│ Roefs      ┆            ┆        

/var/folders/ws/c0kbfc596sgcz0f3y4dzqtyc0000gn/T/ipykernel_48828/2055862598.py:26: DeprecationWarning: The default coalesce behavior of left join will change to `False` in the next breaking release. Pass `coalesce=True` to keep the current behavior and silence this warning.
  errors = errors.join(names, on=["season", "gw", "element"], how="left")


In [75]:
errors.filter(pl.col("bucket_distance") == 2)["position"].value_counts()

position,count
str,u32
"""FWD""",517
"""MID""",2767
"""GK""",535
"""DEF""",2624


In [ ]:
errors.filter((pl.col("bucket_distance") == 2) & (pl.col("position") == "GK")).tail()

season,gw,element,minutes,minutes_bucket,position,value,value_vs_team,pos_value_rank,players_same_pos,predicted_bucket,predicted_confidence,bucket_distance,name,team
str,i64,i64,i64,str,str,i64,f64,i64,i64,str,f64,i32,str,str
"""2025-26""",38,718,0,"""0_minutes""","""MID""",50,0.025893,6,16,"""60_minutes_plus""",0.6935,2,"""Soungoutou Magassa""","""West Ham"""
"""2025-26""",38,722,90,"""60_minutes_plus""","""MID""",50,0.024876,4,21,"""0_minutes""",0.617571,2,"""Florentino Ibrain Morris Luís""","""Burnley"""
"""2025-26""",38,796,81,"""60_minutes_plus""","""MID""",50,0.022212,13,22,"""0_minutes""",0.715539,2,"""Conor Gallagher""","""Spurs"""
"""2025-26""",38,802,0,"""0_minutes""","""MID""",50,0.025893,6,16,"""60_minutes_plus""",0.6935,2,"""Keiber Lamadrid""","""West Ham"""
"""2025-26""",38,807,72,"""60_minutes_plus""","""MID""",54,0.02585,2,21,"""0_minutes""",0.612714,2,"""Rayan Vitor Simplício Rocha""","""Bournemouth"""
